### Testing GraphBuilder

In [1]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from rewiring.model.GCN import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
import torch

dset = datasets.Planetoid(root=f'./../res/datasets/cora', name='cora')


data = dset[0]

train_mask, val_mask, test_mask = TrainValTestSplitter.split('cora', dset)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(dset.num_features, dset.num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()


model.to(device=device)
model.train()
epoch_count = 200
for epoch in range(epoch_count):
    # Clear the gradients
    optimizer.zero_grad()
    # Forward Pass
    out = model(data.x, data.edge_index)
    # Find the Loss
    train_loss = criterion(out[train_mask], data.y[train_mask])    
    # Calculate gradients
    train_loss.backward()
    # Update Weights
    optimizer.step()
    print(f"""Epoch: [{epoch}/{epoch_count}]
    Training loss: {train_loss.item():.4f}""")

    ### VALIDATION
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        val_loss = criterion(out[val_mask], data.y[val_mask])
        _, pred = torch.max(out.data, 1)
        correct = (pred[val_mask] == data.y[val_mask]).float().sum().item()
        acc = correct / val_mask.shape[0]
    
    if (epoch + 1) % 10 == 0:
        print(f"""Epoch: [{epoch+1}/{epoch_count}]
          Training loss: {train_loss.item():.4f},\t Validation loss: {val_loss.item():.4f}
          Accuracy: {acc}""")

Epoch: [0/200]
    Training loss: 1.9223
Epoch: [1/200]
    Training loss: 1.8622
Epoch: [2/200]
    Training loss: 1.7911
Epoch: [3/200]
    Training loss: 1.7237
Epoch: [4/200]
    Training loss: 1.6562
Epoch: [5/200]
    Training loss: 1.5691
Epoch: [6/200]
    Training loss: 1.5100
Epoch: [7/200]
    Training loss: 1.4606
Epoch: [8/200]
    Training loss: 1.3998
Epoch: [9/200]
    Training loss: 1.3287
Epoch: [10/200]
    Training loss: 1.2957
Epoch: [11/200]
    Training loss: 1.2394
Epoch: [12/200]
    Training loss: 1.2056
Epoch: [13/200]
    Training loss: 1.1539
Epoch: [14/200]
    Training loss: 1.1058
Epoch: [15/200]
    Training loss: 1.0633
Epoch: [16/200]
    Training loss: 1.0127
Epoch: [17/200]
    Training loss: 0.9876
Epoch: [18/200]
    Training loss: 0.9379
Epoch: [19/200]
    Training loss: 0.8877
Epoch: [20/200]
    Training loss: 0.8840
Epoch: [21/200]
    Training loss: 0.8470
Epoch: [22/200]
    Training loss: 0.8148
Epoch: [23/200]
    Training loss: 0.7629
Ep

In [ ]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from rewiring.model.GCN import GCN
import torch

data = datasets.TUDataset(root=f'./../res/datasets/proteins', name='PROTEINS')

train_size = int(0.8 * len(data))
test_size = len(data) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(data, [train_size, test_size])

train_dloader: DataLoader = DataLoader(train_dataset, shuffle=True, batch_size=64)
test_dloader: DataLoader = DataLoader(test_dataset, shuffle=False, batch_size=64)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(data.num_features, data.num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

model.to(device=device)
for batch_index, batch in enumerate(train_dloader):
    model.train()
    # Clear the gradients
    optimizer.zero_grad()
    # Forward Pass
    out = model(batch.x, batch.edge_index)
    print(out.shape)
    print(batch.y.shape)
    # Find the Loss
    train_loss = criterion(out, batch.y)    
    # Calculate gradients
    train_loss.backward()

    # Update Weights
    optimizer.step()
    print(f"""Batch: [{batch_index}]
    Training loss: {train_loss.item():.4f}""")
